# Practical 8: Data Manipulation Using Pandas

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

## 1. Load the dataset

The CSV file contains academic performance, demographic, work-experience, placement, and salary information.

In [2]:
from pathlib import Path

file_candidates = [
    Path('Placement_Data_Full_Class.csv'),
    Path('prac8/Placement_Data_Full_Class.csv'),
]
file_path = next(path for path in file_candidates if path.exists())
placements = pd.read_csv(file_path)

placements.head()

,sl_no,gender,ssc_p,ssc_b,hsc_p,hsc_b,hsc_s,degree_p,degree_t,workex,etest_p,specialisation,mba_p,status,salary
0,1,M,67.00,Others,91.00,Others,Commerce,58.00,Sci&Tech,No,55.00,Mkt&HR,58.80,Placed,"270,000.00"
1,2,M,79.33,Central,78.33,Others,Science,77.48,Sci&Tech,Yes,86.50,Mkt&Fin,66.28,Placed,"200,000.00"
2,3,M,65.00,Central,68.00,Central,Arts,64.00,Comm&Mgmt,No,75.00,Mkt&Fin,57.80,Placed,"250,000.00"
3,4,M,56.00,Central,52.00,Central,Science,52.00,Sci&Tech,No,66.00,Mkt&HR,59.43,Not Placed,NaN
4,5,M,85.80,Central,73.60,Central,Commerce,73.30,Comm&Mgmt,No,96.80,Mkt&Fin,55.50,Placed,"425,000.00"


## 2. Understand the data structure

Use `shape`, `columns`, `dtypes`, `info()`, and `head()` to understand the DataFrame before modifying it.

In [3]:
print(f'Rows: {placements.shape[0]}')
print(f'Columns: {placements.shape[1]}')
print('\nColumn names:')
print(placements.columns.tolist())

placements.info()

Rows: 215
Columns: 15

Column names:
['sl_no', 'gender', 'ssc_p', 'ssc_b', 'hsc_p', 'hsc_b', 'hsc_s', 'degree_p', 'degree_t', 'workex', 'etest_p', 'specialisation', 'mba_p', 'status', 'salary']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           215 non-null    int64  
 1   gender          215 non-null    object 
 2   ssc_p           215 non-null    float64
 3   ssc_b           215 non-null    object 
 4   hsc_p           215 non-null    float64
 5   hsc_b           215 non-null    object 
 6   hsc_s           215 non-null    object 
 7   degree_p        215 non-null    float64
 8   degree_t        215 non-null    object 
 9   workex          215 non-null    object 
 10  etest_p         215 non-null    float64
 11  specialisation  215 non-null    object 
 12  mba_p           215 non-null    float64
 13  status          215 n

In [4]:
placements.sample(5, random_state=42)

,sl_no,gender,ssc_p,ssc_b,hsc_p,hsc_b,hsc_s,degree_p,degree_t,workex,etest_p,specialisation,mba_p,status,salary
200,201,M,69.00,Others,60.00,Others,Commerce,65.00,Comm&Mgmt,No,87.55,Mkt&Fin,52.81,Placed,"300,000.00"
212,213,M,67.00,Others,67.00,Others,Commerce,73.00,Comm&Mgmt,Yes,59.00,Mkt&Fin,69.72,Placed,"295,000.00"
138,139,F,82.00,Others,64.00,Others,Science,73.00,Sci&Tech,Yes,96.00,Mkt&Fin,71.77,Placed,"250,000.00"
176,177,F,59.00,Central,60.00,Others,Commerce,56.00,Comm&Mgmt,No,55.00,Mkt&HR,57.90,Placed,"220,000.00"
15,16,F,65.00,Central,75.00,Central,Commerce,69.00,Comm&Mgmt,Yes,72.00,Mkt&Fin,64.66,Placed,"200,000.00"


## 3. Check and clean missing values

First, count missing values in every column. In this dataset, a missing salary is expected for students whose placement status is `Not Placed`; therefore, we will keep those salary values missing instead of replacing them with an arbitrary number.

In [5]:
missing_values = placements.isna().sum().sort_values(ascending=False)
missing_values[missing_values > 0]

salary    67
dtype: int64

In [6]:
placements['salary'] = pd.to_numeric(placements['salary'], errors='coerce')

salary_missing_by_status = placements.groupby('status')['salary'].apply(lambda column: column.isna().sum())
salary_missing_by_status

status
Not Placed    67
Placed         0
Name: salary, dtype: int64

In [7]:
duplicate_count = placements.duplicated().sum()
print(f'Duplicate rows found: {duplicate_count}')

cleaned_placements = placements.drop_duplicates().copy()
print(f'Rows after removing duplicates: {len(cleaned_placements)}')

Duplicate rows found: 0
Rows after removing duplicates: 215


## 5. Select, filter, and sort records

The following examples use Boolean conditions, column selection, and sorting to answer common questions about the data.

## 4. Find and remove duplicate rows

Duplicate records can distort counts and averages. We will check complete-row duplicates and remove them from a cleaned copy.

In [8]:
# Students who were placed and scored at least 70% in their degree
placed_degree_70 = cleaned_placements[
    (cleaned_placements['status'] == 'Placed') &
    (cleaned_placements['degree_p'] >= 70)
][['sl_no', 'gender', 'degree_p', 'specialisation', 'salary', 'status']]

placed_degree_70.head(10)

,sl_no,gender,degree_p,specialisation,salary,status
1,2,M,77.48,Mkt&Fin,"200,000.00",Placed
4,5,M,73.30,Mkt&Fin,"425,000.00",Placed
8,9,M,72.00,Mkt&Fin,"231,000.00",Placed
11,12,M,78.30,Mkt&Fin,"250,000.00",Placed
19,20,M,70.00,Mkt&Fin,"236,000.00",Placed
21,22,F,85.00,Mkt&Fin,"393,000.00",Placed
22,23,F,72.23,Mkt&HR,"360,000.00",Placed
24,25,M,78.86,Mkt&Fin,"360,000.00",Placed
30,31,F,73.00,Mkt&HR,"250,000.00",Placed
33,34,F,81.00,Mkt&Fin,"260,000.00",Placed


In [9]:
# Highest salaries among placed students
highest_salaries = (
    cleaned_placements[cleaned_placements['status'] == 'Placed']
    .sort_values('salary', ascending=False)
    [['sl_no', 'degree_t', 'specialisation', 'salary']]
)

highest_salaries.head(10)

,sl_no,degree_t,specialisation,salary
119,120,Comm&Mgmt,Mkt&Fin,"940,000.00"
150,151,Sci&Tech,Mkt&Fin,"690,000.00"
177,178,Comm&Mgmt,Mkt&Fin,"650,000.00"
77,78,Sci&Tech,Mkt&Fin,"500,000.00"
163,164,Sci&Tech,Mkt&Fin,"500,000.00"
174,175,Sci&Tech,Mkt&Fin,"500,000.00"
53,54,Sci&Tech,Mkt&HR,"450,000.00"
4,5,Comm&Mgmt,Mkt&Fin,"425,000.00"
95,96,Comm&Mgmt,Mkt&Fin,"420,000.00"
39,40,Sci&Tech,Mkt&Fin,"411,000.00"


## 6. Summary functions and descriptive statistics

`describe()` summarizes numerical columns. `value_counts()` summarizes categorical values, while `nunique()` counts distinct values.

In [10]:
numeric_summary = cleaned_placements.describe()
numeric_summary

,sl_no,ssc_p,hsc_p,degree_p,etest_p,mba_p,salary
count,215.00,215.00,215.00,215.00,215.00,215.00,148.00
mean,108.00,67.30,66.33,66.37,72.10,62.28,"288,655.41"
std,62.21,10.83,10.90,7.36,13.28,5.83,"93,457.45"
min,1.00,40.89,37.00,50.00,50.00,51.21,"200,000.00"
25%,54.50,60.60,60.90,61.00,60.00,57.95,"240,000.00"
50%,108.00,67.00,65.00,66.00,71.00,62.00,"265,000.00"
75%,161.50,75.70,73.00,72.00,83.50,66.25,"300,000.00"
max,215.00,89.40,97.70,91.00,98.00,77.89,"940,000.00"


In [11]:
print('Placement status:')
print(cleaned_placements['status'].value_counts())

print('Specialisation:')
print(cleaned_placements['specialisation'].value_counts())

print('Distinct degree types:', cleaned_placements['degree_t'].nunique())

Placement status:
status
Placed        148
Not Placed     67
Name: count, dtype: int64
Specialisation:
specialisation
Mkt&Fin    120
Mkt&HR      95
Name: count, dtype: int64
Distinct degree types: 3


## 7. Grouped analysis

Grouping helps compare placement outcomes across categories. Salary statistics are calculated only for placed students because salary is not applicable to unplaced students.

In [12]:
placement_rate = (
    cleaned_placements.groupby('gender')['status']
    .apply(lambda values: (values == 'Placed').mean())
    .mul(100)
    .round(2)
    .rename('placement_rate_percent')
)
placement_rate

gender
F   63.16
M   71.94
Name: placement_rate_percent, dtype: float64

In [13]:
placed_students = cleaned_placements[cleaned_placements['status'] == 'Placed']
salary_by_specialisation = (
    placed_students.groupby('specialisation')['salary']
    .agg(['count', 'mean', 'median', 'min', 'max'])
    .round(2)
)
salary_by_specialisation

,count,mean,median,min,max
specialisation,,,,,
Mkt&Fin,95,"298,852.63","270,000.00","200,000.00","940,000.00"
Mkt&HR,53,"270,377.36","255,000.00","200,000.00","450,000.00"


In [14]:
work_experience_percentage = (
    cleaned_placements['workex'].eq('Yes').mean() * 100
)
print(f'Students with work experience: {work_experience_percentage:.2f}%')

# Write Pandas statements to answer questions 2-4.

Students with work experience: 34.42%
